<a href="https://colab.research.google.com/github/JulioCesarVilella-Napa/Curriculo/blob/master/APP_gera_RelFotos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

App com interface original

In [2]:
!pip install fpdf Pillow

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=043ab194a9d157d4e19ec2ba46023ea19b285da47fa988cd5cacf904ca86d760
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [3]:
from PyQt6.QtWidgets import (
    QApplication, QMainWindow, QWidget, QVBoxLayout, QHBoxLayout,
    QPushButton, QLineEdit, QLabel, QTextEdit, QProgressBar, QFileDialog,
    QScrollArea, QGridLayout, QSizePolicy
)
from PyQt6.QtGui import QTextCursor, QIcon, QPixmap
from PyQt6.QtCore import QThread, pyqtSignal, Qt
import os
import glob
import datetime

# Novos imports da célula 4fa283f3
import shutil
from PIL import Image
from fpdf import FPDF

# --- Classe Geradora de PDF (adaptada de 4fa283f3) ---
class PDF(FPDF):
    def __init__(self, background_path, pdf_title):
        super().__init__()
        self.background_path = background_path
        self.pdf_title = pdf_title
        self.set_margins(20, 40, 20) # Margens padrão
        self.set_auto_page_break(auto=True, margin=20)
        self.current_y = self.t_margin

    def header(self):
        if self.background_path and os.path.exists(self.background_path):
            self.image(self.background_path, x=0, y=0, w=self.w, h=self.h)
        self.set_font('Arial', 'B', 15)
        self.set_y(10)
        self.cell(0, 10, self.pdf_title, 0, 1, 'R') # Título alinhado à direita
        self.set_y(self.t_margin)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'L')

    def get_image_dimensions(self, path):
        try:
            with Image.open(path) as img:
                return img.size
        except FileNotFoundError:
            return (0, 0)
        except Exception as e:
            return (0, 0)

    def add_centered_photo(self, img_path, legend, max_width, max_height):
        if not os.path.exists(img_path):
            return

        original_width, original_height = self.get_image_dimensions(img_path)
        if original_width == 0 or original_height == 0:
            return

        aspect_ratio = original_width / original_height

        actual_width = max_width
        actual_height = actual_width / aspect_ratio

        if actual_height > max_height:
            actual_height = max_height
            actual_width = actual_height * aspect_ratio

        # Garante que a imagem não exceda a largura máxima
        if actual_width > max_width:
             actual_width = max_width
             actual_height = actual_width / aspect_ratio

        largura_pagina = self.w
        x_centralizado_img = (largura_pagina - actual_width) / 2

        space_after_image = 5
        estimated_legend_height = 8
        space_after_legend = 10

        self.set_y(self.current_y)

        self.image(img_path, x=x_centralizado_img, y=self.current_y, w=actual_width, h=actual_height)

        self.current_y += actual_height + space_after_image
        self.set_y(self.current_y)

        self.set_font('Arial', 'B', 11)
        x_centralizado_legend_cell_start = x_centralizado_img
        self.set_x(x_centralizado_legend_cell_start)
        self.multi_cell(actual_width, estimated_legend_height, legend, 0, 'C')

        self.current_y = self.get_y() + space_after_legend

class Worker(QThread):
    progress = pyqtSignal(int)
    log_msg = pyqtSignal(str)
    finished = pyqtSignal()

    def __init__(self, photos_data, title, bg_path, output_file):
        super().__init__()
        self.photos_data = photos_data # List of {'path': path, 'caption': caption}
        self.title = title
        self.bg_path = bg_path
        self.output_file = output_file

    def run(self):
        try:
            pdf_final = PDF(self.bg_path, self.title)

            # Lógica de paginação adaptada de 4fa283f3
            PHOTOS_ON_FIRST_PAGE = 2
            PHOTOS_ON_SECOND_PAGE = 2
            PHOTOS_ON_SUBSEQUENT_PAGES = 3

            photo_layout_by_page = []
            num_total_photos = len(self.photos_data)
            current_photo_idx = 0

            if num_total_photos > 0:
                page_photos = []
                for _ in range(min(PHOTOS_ON_FIRST_PAGE, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            if num_total_photos > current_photo_idx:
                page_photos = []
                for _ in range(min(PHOTOS_ON_SECOND_PAGE, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            while num_total_photos > current_photo_idx:
                page_photos = []
                for _ in range(min(PHOTOS_ON_SUBSEQUENT_PAGES, num_total_photos - current_photo_idx)):
                    page_photos.append(current_photo_idx)
                    current_photo_idx += 1
                if page_photos:
                    photo_layout_by_page.append(page_photos)

            SPACE_AFTER_IMAGE = 5
            ESTIMATED_LEGEND_HEIGHT = 8
            SPACE_AFTER_LEGEND = 10
            MIN_PHOTO_HEIGHT = 20
            MAX_PHOTO_WIDTH = 170

            total_processed_photos = 0
            for page_number, photo_indices_on_page in enumerate(photo_layout_by_page):
                pdf_final.add_page()
                pdf_final.current_y = pdf_final.t_margin

                num_photos_on_page = len(photo_indices_on_page)
                available_height = pdf_final.h - pdf_final.t_margin - pdf_final.b_margin

                total_fixed_overhead = num_photos_on_page * (SPACE_AFTER_IMAGE + ESTIMATED_LEGEND_HEIGHT + SPACE_AFTER_LEGEND)

                available_height_for_photos = max(0, available_height - total_fixed_overhead)

                max_height_per_photo = max(MIN_PHOTO_HEIGHT, available_height_for_photos / num_photos_on_page if num_photos_on_page > 0 else MIN_PHOTO_HEIGHT)

                for photo_idx_in_data in photo_indices_on_page:
                    photo_data_entry = self.photos_data[photo_idx_in_data]
                    img_path = photo_data_entry['path']
                    legend = photo_data_entry['caption']

                    self.log_msg.emit(f"Adicionando foto à página {page_number + 1}: {os.path.basename(img_path)} com legenda '{legend}'")
                    pdf_final.add_centered_photo(img_path, legend, MAX_PHOTO_WIDTH, max_height_per_photo)
                    total_processed_photos += 1
                    self.progress.emit(int(100 * total_processed_photos / num_total_photos))

            # Garante que o diretório de saída exista
            output_dir = os.path.dirname(self.output_file)
            if output_dir and not os.path.exists(output_dir):
                os.makedirs(output_dir, exist_ok=True)

            pdf_final.output(self.output_file)
            self.log_msg.emit(f'PDF gerado com sucesso em: {self.output_file}')
            self.progress.emit(100)
        except Exception as e:
            self.log_msg.emit(f'Erro na geração do PDF: {str(e)}')
            self.progress.emit(0) # Resetar progresso em caso de erro
        self.finished.emit()


class PhotoWidget(QWidget):
    """Um widget personalizado para exibir uma miniatura de foto e seu índice."""
    def __init__(self, index, image_path, parent=None):
        super().__init__(parent)
        self.setFixedSize(120, 150) # Tamanho fixo para o widget da foto
        self.layout = QVBoxLayout(self)
        self.layout.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.layout.setContentsMargins(0, 0, 0, 0) # Remover margens para aproximar elementos
        self.layout.setSpacing(2) # Reduzir espaçamento entre elementos

        self.image_label = QLabel()
        self.image_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.image_label.setFixedSize(112, 112) # Tamanho fixo para a miniatura (75% de 150x150)

        pixmap = QPixmap(image_path)
        if not pixmap.isNull():
            scaled_pixmap = pixmap.scaled(112, 112, Qt.AspectRatioMode.KeepAspectRatio, Qt.TransformationMode.SmoothTransformation)
            self.image_label.setPixmap(scaled_pixmap)
        else:
            self.image_label.setText("Imagem Inválida")

        self.index_label = QLabel(f"Foto {index}")
        self.index_label.setAlignment(Qt.AlignmentFlag.AlignCenter)
        self.index_label.setStyleSheet("font-weight: bold;")

        self.layout.addWidget(self.image_label)
        self.layout.addWidget(self.index_label)


class MainWindowV2(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Relatório Fotográfico - Versão Galeria')
        self.resize(1200, 900) # Aumentar o tamanho da janela para a galeria
        self.folder_path = None
        self.bg_path = None
        self.photos = [] # Stores {'path': path, 'caption': initial_default_caption}
        self.default_initial_legends = [
            "Logradouro - Lado Direito",
            "Logradouro - Lado Esquerdo",
            "Fachada do Imóvel",
            "Numeração do Imóvel",
            "Detalhes do Imóvel"
        ]

        central = QWidget()
        self.setCentralWidget(central)
        layout = QVBoxLayout(central)

        # Seleção de pasta
        hl_folder = QHBoxLayout()
        hl_folder.addWidget(QLabel('Pasta:'))
        self.folder_label = QLabel('Nenhuma pasta selecionada')
        hl_folder.addWidget(self.folder_label)
        hl_folder.addStretch()
        self.folder_btn = QPushButton('Selecionar Pasta')
        self.folder_btn.clicked.connect(self.select_folder)
        hl_folder.addWidget(self.folder_btn)
        layout.addLayout(hl_folder)

        # Título
        layout.addWidget(QLabel('Título do PDF:'))
        self.title_edit = QLineEdit('Relatório Fotográfico')
        layout.addWidget(self.title_edit)

        # Imagem de fundo
        hl_bg = QHBoxLayout()
        hl_bg.addWidget(QLabel('Fundo:'))
        self.bg_label = QLabel('Nenhuma')
        hl_bg.addWidget(self.bg_label)
        self.bg_btn = QPushButton('Selecionar Imagem de Fundo')
        self.bg_btn.clicked.connect(self.select_bg)
        hl_bg.addWidget(self.bg_btn)
        layout.addLayout(hl_bg)

        # Área de rolagem para as fotos (substitui a tabela)
        self.scroll_area = QScrollArea()
        self.scroll_area.setWidgetResizable(True)

        self.photos_grid_widget = QWidget() # Widget container para o grid
        self.photos_grid_layout = QGridLayout(self.photos_grid_widget)
        self.photos_grid_layout.setAlignment(Qt.AlignmentFlag.AlignLeft | Qt.AlignmentFlag.AlignTop)
        self.scroll_area.setWidget(self.photos_grid_widget)

        layout.addWidget(self.scroll_area)

        # Entrada numérica para ordem das fotos
        hl_order_input = QHBoxLayout()
        hl_order_input.addWidget(QLabel('Ordem das Fotos (Ex: 3,4,1):'))
        self.order_input = QLineEdit()
        self.order_input.setPlaceholderText('Entre com os números das fotos separados por vírgula')
        hl_order_input.addWidget(self.order_input)
        layout.addLayout(hl_order_input)

        # Botão Gerar PDF
        hl_generate = QHBoxLayout()
        hl_generate.addStretch()
        self.gen_btn = QPushButton('Gerar PDF')
        self.gen_btn.clicked.connect(self.generate_pdf)
        hl_generate.addWidget(self.gen_btn)
        layout.addLayout(hl_generate)

        # Progresso
        self.progress = QProgressBar()
        self.progress.setRange(0, 100)
        self.progress.setValue(0)
        layout.addWidget(self.progress)

        # Log
        layout.addWidget(QLabel('Log:'))
        self.log = QTextEdit()
        self.log.setReadOnly(True)
        self.log.setFixedHeight(100) # Reduzir a altura da caixa de log
        layout.addWidget(self.log)

    def select_folder(self):
        path = QFileDialog.getExistingDirectory(self, 'Selecionar Pasta de Fotos')
        if path:
            self.folder_path = path
            self.folder_label.setText(path)
            self.load_photos()

    def load_photos(self):
        if not self.folder_path:
            return
        exts = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff']
        images = []
        for ext in exts:
            images.extend(glob.glob(os.path.join(self.folder_path, ext)))
        images.sort() # Ordenar as imagens por nome de arquivo

        new_photos_data = []
        for i, img_path in enumerate(images):
            caption = ""
            # A legenda inicial não é mais atribuída aqui para exibição,
            # apenas para manter a estrutura de dados original, se necessário futuramente.
            new_photos_data.append({'path': img_path, 'caption': caption})

        self.photos = new_photos_data
        self.populate_photos_grid() # Chamar o novo método de preenchimento da grade
        self.log_message(f'Carregadas {len(images)} fotos da pasta {self.folder_path}')

    def clear_layout(self, layout):
        """Limpa todos os widgets de um layout."""
        if layout is not None:
            while layout.count():
                item = layout.takeAt(0)
                widget = item.widget()
                if widget is not None:
                    widget.deleteLater()
                else:
                    self.clear_layout(item.layout())

    def populate_photos_grid(self):
        self.clear_layout(self.photos_grid_layout) # Limpar a grade antes de preencher

        if not self.photos: # Se não houver fotos, não faz nada
            return

        # Definir o número de colunas para a grade
        num_columns = 4
        current_row = 0
        current_col = 0

        for i, photo_entry in enumerate(self.photos):
            photo_widget = PhotoWidget(i + 1, photo_entry['path']) # Passar índice 1-based e caminho
            self.photos_grid_layout.addWidget(photo_widget, current_row, current_col)

            current_col += 1
            if current_col >= num_columns:
                current_col = 0
                current_row += 1

        # Ajustar o tamanho do widget que contém o grid para que a scroll area funcione corretamente
        self.photos_grid_widget.adjustSize()

    def select_bg(self):
        file, _ = QFileDialog.getOpenFileName(self, 'Imagem de Fundo', '', 'Imagens (*.jpg *.jpeg *.png)')
        if file:
            self.bg_path = file
            self.bg_label.setText(os.path.basename(file))

    def generate_pdf(self):
        # 1. Parse user's numerical order input
        order_text = self.order_input.text().strip()
        if not order_text:
            QMessageBox.warning(self, 'Aviso', 'Por favor, insira a ordem das fotos (números separados por vírgula).')
            return

        selected_original_indices = []
        try:
            # Convert 1-based user input to 0-based list indices
            selected_indices_1_based = [int(idx.strip()) for idx in order_text.split(',') if idx.strip()]
            for idx_1_based in selected_indices_1_based:
                idx_0_based = idx_1_based - 1
                if not (0 <= idx_0_based < len(self.photos)):
                    QMessageBox.warning(self, 'Erro', f'Número de foto inválido: {idx_1_based}. Por favor, insira números entre 1 e {len(self.photos)}.')
                    return
                selected_original_indices.append(idx_0_based)
        except ValueError:
            QMessageBox.warning(self, 'Erro', 'Entrada de ordem inválida. Use números separados por vírgula (Ex: 3,4,1).')
            return

        # 2. Construct the photos_data list for the Worker based on the chosen order
        photos_data_for_worker = []
        for i, original_photo_idx in enumerate(selected_original_indices):
            photo_entry = self.photos[original_photo_idx] # Get the original photo entry
            # Assign caption based on its position in the *newly ordered list for the PDF*
            caption = ""
            if i < len(self.default_initial_legends):
                caption = self.default_initial_legends[i]
            else:
                caption = "Detalhes do Imóvel" # Default for photos beyond the initial set
            photos_data_for_worker.append({'path': photo_entry['path'], 'caption': caption})

        if not photos_data_for_worker:
            QMessageBox.warning(self, 'Aviso', 'Nenhuma foto selecionada na ordem fornecida.')
            return

        # 3. Proceed with PDF generation using photos_data_for_worker
        timestamp = datetime.datetime.now().strftime("_%Y%m%d_%H%M%S")
        suggested_filename = os.path.join(self.folder_path or '', f'relatorio_{timestamp}.pdf')

        output_file, _ = QFileDialog.getSaveFileName(self, 'Salvar PDF', suggested_filename, 'PDF (*.pdf)')
        if not output_file:
            return

        self.gen_btn.setEnabled(False)
        self.progress.setValue(0)
        self.worker = Worker(photos_data_for_worker, self.title_edit.text(), self.bg_path, output_file) # Pass the newly constructed list
        self.worker.progress.connect(self.progress.setValue)
        self.worker.log_msg.connect(self.log_message)
        self.worker.finished.connect(self.on_worker_finished)
        self.worker.start()

    def log_message(self, msg):
        cursor = self.log.textCursor()
        cursor.movePosition(QTextCursor.MoveOperation.End)
        self.log.setTextCursor(cursor)
        self.log.insertPlainText(msg + '\n')
        self.log.ensureCursorVisible()

    def on_worker_finished(self):
        self.gen_btn.setEnabled(True)
        self.progress.setValue(100)
        self.log_message('Processamento concluído.')

# Para executar esta nova versão, descomente as linhas abaixo:
# if __name__ == '__main__':
#     app = QApplication([])
#     win = MainWindowV2()
#     win.show()
#     app.exec()